# 00 · Connect your data — files, a Google Sheet, a database

**Use case.** Before anything else, the data has to get in. This chapter shows the three ways, with a fresh project for each, so every step runs for real: **upload** files, **link** a Google Sheet that stays live, and (as code you can copy) a **database connector**. The other chapters reuse projects; this one starts from nothing on purpose.

**What you will learn**
1. Create a project and upload CSV files — presign → PUT → confirm; rows counted server-side, plan caps enforced
2. Let Langsat detect keys and relationships, then read the sources back (`sources.list()`)
3. Link a Google Sheet: preview its tabs, import them, re-pull with `refresh()`, put it on a schedule
4. Private sheets: share with the Langsat service account, nothing else changes
5. A Postgres / MySQL connector on the Team plan (`ls.connectors`) — the calls, without a live database
6. Delete a project when you are done

**What this costs.** nothing — uploads, links, schema detection and a refresh under 500K rows are free; the two projects are deleted at the end.

> Every cell below ran for real against `api.langsat.ai` — the outputs are what the API returned. Re-running is safe:
> projects are found by name and reused, and a finished model is not retrained.

## Three ways in

| way | call | when |
|---|---|---|
| **Upload** | `p.sources.upload(*paths)` — CSV, Parquet, one-sheet XLSX | a one-off export; the data changes when you re-upload |
| **Link a Google Sheet** | `p.sources.preview_link(url)` → `p.sources.link(url, tabs=[…])` | a sheet people keep editing; `refresh()` re-pulls it, a schedule does it for you |
| **Database connector** (Team) | `ls.connectors.create(…)` → `test` → `sync` | Postgres / MySQL tables synced into the project on a schedule |

Whichever way the rows arrive, everything after is identical: schema detection, cleaning, dashboards, chat, training. A project has a **kind** — `data_analysis` (dashboards, chat, forecasts; cleaning is a step) or `data_science` (the same plus training; the training pipeline cleans).

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show, fig, metrics_table, project_models
from langsat import viz
import pandas as pd

ls = connect()
SHEET_URL = "https://docs.google.com/spreadsheets/d/1mLVFDGYHdDfNOOEvE5mpYiDgYH7nxkveZ3L00CoaBD8/edit"   # a public copy of the same three tables, one tab each

signed in as team@langsat.ai · tier team · key 'TEST-SDK-4' with 20 scopes


## Way 1 · Upload files

`sources.upload(*paths)` is three HTTP calls the SDK makes for you: a **presigned URL** per file, a **PUT** straight to S3 (the bytes never pass through the API), then a **confirm** on which the server counts rows and columns and names the table after the file. Row caps and plan limits are enforced at the confirm — an upload over the cap is refused before anything is stored. Under a team, a new project lands in the team's org automatically.

In [2]:
from _common import DATA_DIR
NAME = "amazon-reviews-connect-upload"
for old in [x for x in ls.projects.list() if x.name == NAME]:
    old.delete(); print("deleted the previous", NAME, "— starting from nothing")
p = ls.projects.create(NAME, kind="data_analysis")
print("created", p.id, "· kind", p.data.get("project_type"), "· status", p.status)

up = p.sources.upload(DATA_DIR / "customer.csv", DATA_DIR / "product.csv", DATA_DIR / "review.csv")
for f in up["files"]:
    print(f"  {f['name']:<13} {f['rows']:>7,} rows × {f['cols']} columns  ({f['size_bytes'] / 1e6:.1f} MB) → table '{f['table'] or f['name'][:-4]}'")
print("status:", p.refresh().status, "· files:", p.data["file_names"])

created 5423ea43-8ef2-436f-b254-4e935d57d1c4 · kind data_analysis · status created


  customer.csv    9,821 rows × 2 columns  (0.2 MB) → table 'customer'
  product.csv     8,607 rows × 6 columns  (7.0 MB) → table 'product'
  review.csv     10,000 rows × 7 columns  (5.7 MB) → table 'review'


status: uploaded · files: ['customer.csv', 'product.csv', 'review.csv']


### Detect the schema

Long actions return a **Job**; `wait()` polls and raises `JobFailed` / `JobTimeout`. Schema detection samples every table and finds primary keys, foreign keys and the time column — nothing is declared by you. Then `sources.list()` shows each source as the platform holds it: its id (what `refresh`, `schedule` and `delete` take), type, rows, and when it was last pulled.

In [3]:
job = p.schema.detect()
job.wait(timeout=900, on_progress=lambda j: print("  ", j.get("stage"), j.get("progress_pct"), "%"))
sr = print_schema(p)
pd.DataFrame(p.sources.list())[["file_id", "name", "source_type", "row_count", "column_count", "cleaning_status"]]

   loading_data 5.0 %


   loading_data 15.0 %


   loading_data 35.0 %


   complete 100.0 %


primary keys : {'review': '__row_id__', 'product': 'product_id', 'customer': 'customer_id'}
foreign keys : {'review': {'product_id': 'product', 'customer_id': 'customer'}, 'product': {}, 'customer': {}}
time columns : {'review': 'review_time', 'product': None, 'customer': None}


  customer: 9,821 rows · 2 columns
  product: 8,607 rows · 6 columns
  review: 10,000 rows · 7 columns


,file_id,name,source_type,row_count,column_count,cleaning_status
0,73a24444-44b7-4e74-a417-8f7a3c287400,customer.csv,upload,9821,2,pending
1,81425972-95dc-434e-8528-c42be29165a5,product.csv,upload,8607,6,pending
2,a7c3454e-e4eb-40a6-89e3-61429a0387ca,review.csv,upload,10000,7,pending


### Re-upload and a fourth file

A second `upload()` of a file with the same name **replaces** that table (and invalidates what was built on it — the app asks before it does that; the API just does it). A file with a new name adds a table. `estimates()` tells you what the next paid step would cost — cleaning is free here.

In [4]:
print("estimates:", {k: p.estimates()[k] for k in ("clean", "ask", "predict")})
print("cleaning is a separate step on a data_analysis project — chapter 01 runs it: p.cleaning.clean().wait()")

estimates: {'clean': {'credits': 0, 'lane': 'lambda'}, 'ask': {'ai_questions': 1, 'quota': {'available': True, 'pct_today': 1, 'pct_month': 0, 'reason': None}}, 'predict': {'credits_per_call': 50}}
cleaning is a separate step on a data_analysis project — chapter 01 runs it: p.cleaning.clean().wait()


## Way 2 · Link a Google Sheet

A sheet that is *Anyone with the link → Viewer* links as it is. A **private** sheet: share it (Viewer) with the Langsat service account — `langsat-sheets@langsat-sheets.iam.gserviceaccount.com` — and link the same URL; the platform reads it through the Sheets API, live, no export cache. `preview_link` costs nothing and writes nothing: it lists the workbook's tabs with row and column counts so you can pick which become tables.

In [5]:
NAME = "amazon-reviews-connect-sheet"
for old in [x for x in ls.projects.list() if x.name == NAME]:
    old.delete(); print("deleted the previous", NAME)
p2 = ls.projects.create(NAME, kind="data_analysis")

preview = p2.sources.preview_link(SHEET_URL)
print("workbook", preview["doc_id"])
pd.DataFrame(preview["tabs"])

workbook 1mLVFDGYHdDfNOOEvE5mpYiDgYH7nxkveZ3L00CoaBD8


,sheet_name,cols,visible,rows,importable
0,customer,2,True,10000,True
1,product,6,True,10000,True
2,review,7,True,10000,True


In [6]:
linked = p2.sources.link(SHEET_URL, tabs=["customer", "product", "review"])
print(json.dumps(linked, indent=1, default=str)[:600])
p2.schema.detect().wait(timeout=900)
p2.refresh()
print("status:", p2.status, "· tables:", [(t["name"], t["rows"]) for t in p2.tables.list()])

{
 "sources": [
  {
   "file_id": "7206a36e-2e3b-4cb3-81cb-10fcfb7e2a39",
   "name": "customer",
   "source_type": "gsheet",
   "is_live": true,
   "source_url": "https://docs.google.com/spreadsheets/d/1mLVFDGYHdDfNOOEvE5mpYiDgYH7nxkveZ3L00CoaBD8/export?format=xlsx",
   "refresh_schedule": "manual",
   "next_refresh_at": null,
   "last_refreshed_at": null,
   "refresh_status": null,
   "refresh_error": null,
   "row_count": 10000,
   "column_count": 2,
   "cleaning_status": "pending",
   "provider": "gsheet",
   "doc_id": "1mLVFDGYHdDfNOOEvE5mpYiDgYH7nxkveZ3L00CoaBD8",
   "sheet_name": "custom


status: schema_done · tables: [('customer', 1), ('product', 1), ('review', 1)]


### Keep it fresh

`refresh()` re-pulls the sheet, re-cleans and re-renders every recipe chart on the project's dashboards — a no-op when the bytes have not changed (`force=True` to insist). It is priced by lane: 0 credits under 500K rows. `schedule()` makes the platform do it on its own (`15min` · `hourly` · `daily` · `weekly`; the plan decides which are allowed); `set_live` turns a source's live re-pull on and off.

In [7]:
src = pd.DataFrame(p2.sources.list())
display(src[["file_id", "name", "source_type", "sheet_name", "row_count", "is_live", "refresh_schedule", "last_refreshed_at"]])
fid = src.iloc[0]["file_id"]
job = p2.sources.refresh(fid)
job.wait(timeout=1800)
print("refresh:", job.status, "·", (job.raw or {}).get("message"))
p2.sources.schedule(fid, "daily")
print("schedule:", next(x for x in p2.sources.list() if x["file_id"] == fid)["refresh_schedule"], "· next run:", next(x for x in p2.sources.list() if x["file_id"] == fid)["next_refresh_at"])

,file_id,name,source_type,sheet_name,row_count,is_live,refresh_schedule,last_refreshed_at
0,7206a36e-2e3b-4cb3-81cb-10fcfb7e2a39,customer,gsheet,customer,10000,True,manual,None
1,9015aae0-a186-455e-b1ef-1009b7bcf272,product,gsheet,product,10000,True,manual,None
2,4c9fe6b4-e4c3-46b5-a65e-f0f406193038,review,gsheet,review,10000,True,manual,None


refresh: succeeded · Refresh started.


schedule: daily · next run: 2026-09-16T03:13:43.720209


## Way 3 · A database connector (Team plan)

The connector syncs tables from Postgres or MySQL into the project — on demand or on a schedule — over TLS from fixed egress IPs you can allow-list. It needs a reachable database, so it is shown here as the calls, not run:

```python
ls.connectors.allowlist_ips()                              # the IPs to allow on your database's firewall
c = ls.connectors.create(project_id=p.id, display_name="orders-db", db_type="postgres",
                         host="db.example.com", port=5432, database_name="shop", schema_name="public",
                         username="langsat_ro", password="…", ssl_enabled=True)
ls.connectors.test(c["connection_id"])                     # connectivity + permissions, nothing copied
ls.connectors.preview_table(c["connection_id"], "orders")  # 20 rows
ls.connectors.sync_estimate(c["connection_id"])            # rows, bytes, credits before you commit
ls.connectors.sync(c["connection_id"])                     # → the tables land in the project; schedule as with a sheet
```

Credentials are stored encrypted and never returned by the API.

## Clean up

Deleting a project removes its storage, dashboards, models and shares. Both projects here were made only to show the steps; the chapters that follow use `amazon-reviews-explore` and friends.

In [8]:
for x in (p, p2):
    x.delete(); print("deleted", x.name)
save_metrics(".", {"notebook": "00_connect_your_data", "task": "connect data · upload, Google Sheet, connector", "model": "—",
                   "headline": {"uploaded_rows": {f["name"][:-4]: f["rows"] for f in up["files"]},
                                "sheet_tabs": [t["sheet_name"] for t in preview["tabs"]], "refresh": job.status}})

deleted amazon-reviews-connect-upload


deleted amazon-reviews-connect-sheet
wrote results/metrics.json


PosixPath('results/metrics.json')

## Where to go next

- [01 · Set up and explore](../01_setup_and_explore/) — clean, chart, publish, ask
- Docs: https://langsat.ai/resources/learn/getting-started/sdk